In [5]:
### VERY IMPORTANT!!!
# create a .venv and install dependencies
# Run in your terminal: "python -m pip install pandas ipykernel numpy jupyter"

In [7]:
# import sys
# print(sys.executable)

In [3]:
import pandas as pd
import numpy as np

In [ ]:

gravity_base = pd.read_csv("../outputs_csv/gravity_base.csv")



/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_64747/1761165302.py:1: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  gravity_base = pd.read_csv("../outputs_csv/gravity_base.csv")


In [4]:
gravity_base['dropBackType'].value_counts()

dropBackType
Traditional               5645596
Scramble                  1180300
Unknown                    532224
Designed Rollout Right     259886
Scramble Rollout Right     159126
Designed Rollout Left      142736
Scramble Rollout Left       28578
Designed Run                 4246
Name: count, dtype: int64

All plays with dropBackType = "Unknown" are plays that were nullified by penalty. These plays can be dropped. Screens and RPOs were automatically dropped by Kaggle.

In [5]:
# exclude rollouts, designed runs, and plays nullified by penalty (Unknown)
base_filtered = gravity_base[(gravity_base['dropBackType'] == 'Traditional') | (gravity_base['dropBackType'] == 'Scramble')]

In [6]:
each_play = base_filtered.dropna(subset='event')
each_play=each_play.drop_duplicates(subset=['gameId', 'playId', 'frameID'])
each_play['event'].value_counts()

event
ball_snap                    7417
pass_forward                 6570
autoevent_ballsnap           3279
autoevent_passforward        3250
play_action                  1351
qb_sack                       412
run                           378
pass_arrived                  319
autoevent_passinterrupted     166
man_in_motion                 138
line_set                      121
shift                         101
pass_tipped                    96
first_contact                  68
qb_strip_sack                  57
pass_outcome_incomplete        34
pass_outcome_caught            20
fumble                         12
handoff                        10
fumble_offense_recovered        7
huddle_break_offense            2
tackle                          2
dropped_pass                    1
penalty_flag                    1
Name: count, dtype: int64

In [7]:
each_play = base_filtered.dropna(subset='event')
each_play=each_play.drop_duplicates(subset=['gameId', 'playId', 'frameID'])

# get the frames where the window ends. Window ends when a forward pass is thrown, 
# or when a QB gets sacked, or when a QB gets strip-sacked, or when a QB takes off to run.
end_window_event = each_play[(each_play['event'] == 'pass_forward') | (each_play['event'] == 'qb_sack')
| (each_play['event'] == 'qb_strip_sack') | (each_play['event'] == 'run')][['gameId', 'playId', 'frameID']]
end_window_event = end_window_event.sort_values(by=['gameId', 'playId', 'frameID'])
end_window_event = end_window_event.drop_duplicates(subset=['gameId', 'playId'], keep='first')

# get the frames where the ball is snapped
ball_snaps = each_play[each_play['event'] == 'ball_snap'][['gameId', 'playId','frameID']]
ball_snaps = ball_snaps.drop_duplicates(subset=['gameId', 'playId'], keep='first')

# get all plays where a ball_snapped frame and an end_window frame exists
each_play=each_play.merge(end_window_event[['gameId','playId']], on=['gameId','playId'],how='inner')
each_play=each_play.merge(ball_snaps[['gameId','playId']], on=['gameId','playId'],how='inner')
merged_temp = pd.concat([end_window_event, ball_snaps])
each_play=each_play.merge(merged_temp[['gameId','playId','frameID']], on=['gameId','playId','frameID'], how='inner')

In [8]:
# confirm that each play has two frames: a ball_snap frame and an end_window frame
value_counts = each_play.groupby(['gameId','playId'])['event'].count()
value_counts.value_counts()


event
2    7386
Name: count, dtype: int64

In [9]:
temp = base_filtered.merge(each_play[['gameId','playId']].drop_duplicates(), on=['gameId','playId'], how='inner')

ball_snaps = each_play[each_play['event'] == 'ball_snap']
ball_snaps['frameIdBallSnap'] = ball_snaps['frameID']
end_window_event = each_play[each_play['event'] != 'ball_snap']
end_window_event['frameIdEndWindow'] = end_window_event['frameID']

temp = temp.merge(ball_snaps[['gameId','playId','frameIdBallSnap']], on=['gameId','playId'], how='inner')
temp = temp.merge(end_window_event[['gameId','playId','frameIdEndWindow']], on=['gameId','playId'], how='inner')


temp['three_seconds_after'] = temp['frameIdBallSnap']+30
temp['frameIdEndWindow'] = temp[['frameIdEndWindow', 'three_seconds_after']].min(axis=1)

# filter to an upper bound of the frameIdEndWindow
temp = temp[(temp['frameID'] <= temp['frameIdEndWindow'])]

In [10]:
# FILTER OUT FRAMES WHERE THE QB IS OUTSIDE THE TACKLE BOX

# --------------------------------------------------
# 1. Standardize key columns if needed
# --------------------------------------------------
# Adjust these names if your actual columns differ
GAME_COL = "gameId"
PLAY_COL = "playId"
FRAME_COL = "frameID"
EVENT_COL = "event"
POSITION_COL = "pff_positionLinedUp"
# PLAYER_COL = "displayName"   # or whatever identifies QB if needed
Y_COL = "y"

# # Optional: normalize string columns to avoid case issues
# df[EVENT_COL] = df[EVENT_COL].astype(str).str.lower().str.strip()
# df[POSITION_COL] = df[POSITION_COL].astype(str).str.upper().str.strip()

# --------------------------------------------------
# 2. Get LT y-position at ball_snap for each play
# --------------------------------------------------
lt_snap = (
    temp[
        (temp[EVENT_COL] == "ball_snap") &
        (temp[POSITION_COL] == "LT")
    ][[GAME_COL, PLAY_COL, Y_COL]]
    .drop_duplicates(subset=[GAME_COL, PLAY_COL])
    .rename(columns={Y_COL: "y_LT"})
)

# --------------------------------------------------
# 3. Get RT y-position at ball_snap for each play
# --------------------------------------------------
rt_snap = (
    temp[
        (temp[EVENT_COL] == "ball_snap") &
        (temp[POSITION_COL] == "RT")
    ][[GAME_COL, PLAY_COL, Y_COL]]
    .drop_duplicates(subset=[GAME_COL, PLAY_COL])
    .rename(columns={Y_COL: "y_RT"})
)

# --------------------------------------------------
# 4. Merge y_LT and y_RT onto the full tracking dataframe
# --------------------------------------------------
df_with_tackles = temp.merge(
    lt_snap,
    on=[GAME_COL, PLAY_COL],
    how="inner"
).merge(
    rt_snap,
    on=[GAME_COL, PLAY_COL],
    how="inner"
)

# --------------------------------------------------
# 5. Identify QB rows
# --------------------------------------------------
# Best if you already have a position column for player role on each row.
# Replace this with your actual QB-identifying column if needed.
qb_rows = df_with_tackles[POSITION_COL].astype(str).str.upper().eq("QB")

qb_tracking = df_with_tackles[qb_rows].copy()

# --------------------------------------------------
# 6. Define tackle box boundaries
# --------------------------------------------------
# Since one tackle may have a lower y and the other higher y,
# use min/max to avoid assuming LT < RT or RT < LT.
qb_tracking["tackle_box_min_y"] = qb_tracking[["y_LT", "y_RT"]].min(axis=1)
qb_tracking["tackle_box_max_y"] = qb_tracking[["y_LT", "y_RT"]].max(axis=1)

# --------------------------------------------------
# 7. Keep only QB frames where QB is inside tackle box
# --------------------------------------------------
qb_in_tackle_box = qb_tracking[
    (qb_tracking[Y_COL] >= qb_tracking["tackle_box_min_y"]) &
    (qb_tracking[Y_COL] <= qb_tracking["tackle_box_max_y"])
].copy()

# --------------------------------------------------
# 8. If you want ALL player rows for only those frames where QB is in tackle box
# --------------------------------------------------
valid_qb_frames = qb_in_tackle_box[[GAME_COL, PLAY_COL, FRAME_COL]].drop_duplicates()

tracking_only_when_qb_in_tackle_box = df_with_tackles.merge(
    valid_qb_frames,
    on=[GAME_COL, PLAY_COL, FRAME_COL],
    how="inner"
)

In [11]:
# filter to a lower bound of 5 frames after ball snap
filtered_lower_and_upper_bound = tracking_only_when_qb_in_tackle_box[(tracking_only_when_qb_in_tackle_box['frameID'] >= tracking_only_when_qb_in_tackle_box['frameIdBallSnap']+5)]

max_upper_bound = (
    filtered_lower_and_upper_bound
    .groupby(['gameId', 'playId'])['frameID']
    .max()
    .reset_index()
    .rename(columns={'frameID': 'maxUpperBound'})
)

filtered_lower_and_upper_bound = filtered_lower_and_upper_bound.merge(
    max_upper_bound,
    on=['gameId', 'playId'],
    how='inner'
)
filtered_lower_and_upper_bound['frameIdEndWindow'] = filtered_lower_and_upper_bound['maxUpperBound']
filtered_lower_and_upper_bound = filtered_lower_and_upper_bound.drop(columns=['maxUpperBound'])

In [12]:
# confirm that the longest frame window is 2.5 seconds long.

(filtered_lower_and_upper_bound.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - filtered_lower_and_upper_bound.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].max()

np.int64(25)

In [13]:
# filter out plays where the frameIdEndWindow - frameIdBallSnap <= 15

play_windows = (
    filtered_lower_and_upper_bound[['gameId', 'playId', 'frameIdBallSnap', 'frameIdEndWindow']]
    .drop_duplicates()
)

play_windows['window_length'] = (
    play_windows['frameIdEndWindow'] - play_windows['frameIdBallSnap']
)

valid_plays = play_windows[play_windows['window_length'] > 15][['gameId', 'playId']]

tracking_filtered_by_window_length = filtered_lower_and_upper_bound.merge(
    valid_plays,
    on=['gameId', 'playId'],
    how='inner'
)
tracking_filtered_by_window_length

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,a,dis,o,dir,event,frameIdBallSnap,frameIdEndWindow,three_seconds_after,y_LT,y_RT
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.100,...,1.61,0.13,119.88,261.95,NaN,6,36,36,26.92,21.31
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.200,...,1.43,0.15,113.65,260.27,NaN,6,36,36,26.92,21.31
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.300,...,1.37,0.19,107.99,261.01,NaN,6,36,36,26.92,21.31
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.400,...,1.63,0.25,114.80,264.15,NaN,6,36,36,26.92,21.31
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.500,...,1.28,0.23,119.55,263.17,NaN,6,36,36,26.92,21.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3562587,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,0.28,0.74,48.35,60.83,NaN,7,37,37,26.85,20.70
3562588,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,0.67,0.75,48.35,61.74,NaN,7,37,37,26.85,20.70
3562589,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,1.43,0.75,58.99,63.79,NaN,7,37,37,26.85,20.70
3562590,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,2.01,0.75,63.66,65.69,NaN,7,37,37,26.85,20.70


In [14]:
# check if i filtered frames correctly


check_df = (
    tracking_filtered_by_window_length
    .groupby(['gameId', 'playId'], as_index=False)['frameID']
    .min()
    .merge(
        tracking_filtered_by_window_length[['gameId', 'playId', 'frameIdBallSnap']].drop_duplicates(),
        on=['gameId', 'playId'],
        how='left'
    )
)

check_df['matches'] = check_df['frameID'] == check_df['frameIdBallSnap'] + 5
print((check_df['matches'] == False).sum())

print((tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].min())

print((tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].max())


# check if QB is within tackle box at all times
qbs_only = tracking_filtered_by_window_length[tracking_filtered_by_window_length['pff_positionLinedUp'] == 'QB']
sum_outside_tacklebox = ((
    (qbs_only['y'] >= qbs_only[['y_LT', 'y_RT']].min(axis=1)) &
    (qbs_only['y'] <= qbs_only[['y_LT', 'y_RT']].max(axis=1))
) == False).sum()
print(sum_outside_tacklebox)


0
11
25
0


In [15]:
# tracking_filtered_by_window_length.to_csv("../outputs_csv/tracking_filtered_by_play_and_frame.csv", index=False)

In [16]:
base_filtered = base_filtered.merge(
    play_windows[['gameId','playId']],
    on=['gameId','playId'],
    how='inner'
)

base_filtered = base_filtered.merge(
    tracking_filtered_by_window_length.drop_duplicates(['gameId','playId'])[['gameId','playId','frameIdEndWindow']], 
    on=['gameId','playId'],
    how='inner'
)

base_filtered = base_filtered[
    base_filtered['frameID'] <= base_filtered['frameIdEndWindow']
]

base_filtered

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.200,...,right,37.78,24.22,0.23,0.11,0.02,164.33,92.87,NaN,36
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.300,...,right,37.78,24.24,0.16,0.10,0.01,160.24,68.55,NaN,36
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.400,...,right,37.73,24.25,0.15,0.24,0.06,152.13,296.85,NaN,36
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.500,...,right,37.69,24.26,0.25,0.18,0.04,148.33,287.55,NaN,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6739234,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,right,40.36,44.25,7.47,0.28,0.74,48.35,60.83,NaN,37
6739235,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,right,41.02,44.60,7.45,0.67,0.75,48.35,61.74,NaN,37
6739236,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,right,41.68,44.94,7.48,1.43,0.75,58.99,63.79,NaN,37
6739237,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,right,42.36,45.25,7.41,2.01,0.75,63.66,65.69,NaN,37


In [17]:
base_filtered.drop_duplicates(subset=['gameId','playId'])

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
946,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:09.900,...,left,106.92,20.83,0.07,0.08,0.01,153.18,250.65,NaN,32
1760,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.000,...,left,75.23,28.60,0.27,0.54,0.03,131.50,46.10,line_set,28
2442,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:51.600,...,left,48.71,28.12,0.20,0.19,0.03,88.15,43.07,NaN,36
3388,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.000,...,left,53.50,36.53,0.03,0.03,0.03,85.90,72.33,NaN,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6733628,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.300,...,left,18.90,47.67,0.00,0.00,0.00,241.60,341.47,NaN,37
6734948,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:40.400,...,right,33.29,26.85,0.00,0.00,0.00,86.74,34.36,NaN,37
6736004,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.300,...,right,36.31,20.84,0.00,0.00,0.00,89.16,353.34,NaN,37
6737170,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:39.600,...,right,27.97,20.56,0.01,0.01,0.01,92.81,250.80,NaN,33


In [18]:
# base_filtered.to_csv("../outputs_csv/base_filtered.csv", index=False)

In [20]:
filtered_df = tracking_filtered_by_window_length
filtered_df = filtered_df.drop(['y_LT','y_RT','three_seconds_after','ttt_le_1_5_and_behind_los'],axis=1)
pass_blockers = filtered_df[~filtered_df['pff_blockType'].isna()]
pass_rushers = filtered_df[filtered_df['pff_role'] == 'Pass Rush']

In [21]:
# calculate weighted distances

blockers = pass_blockers[
    ['gameId', 'playId', 'frameID', 'nflId', 'x', 'y', 'o']
].copy()

rushers = pass_rushers[
    ['gameId', 'playId', 'frameID', 'nflId', 'x', 'y']
].copy()

blockers = blockers.rename(columns={
    'nflId': 'blocker_nflId',
    'x': 'x_blocker',
    'y': 'y_blocker',
    'o': 'o_blocker'
})

rushers = rushers.rename(columns={
    'nflId': 'rusher_nflId',
    'x': 'x_rusher',
    'y': 'y_rusher'
})

# Every blocker paired with every rusher in the same frame
blocker_rusher_pairs = blockers.merge(
    rushers,
    on=['gameId', 'playId', 'frameID'],
    how='inner'
)

# Vector from blocker to rusher
blocker_rusher_pairs['dx'] = (
    blocker_rusher_pairs['x_rusher'] - blocker_rusher_pairs['x_blocker']
)
blocker_rusher_pairs['dy'] = (
    blocker_rusher_pairs['y_rusher'] - blocker_rusher_pairs['y_blocker']
)

# Euclidean distance
blocker_rusher_pairs['actual_distance'] = np.sqrt(
    blocker_rusher_pairs['dx']**2 + blocker_rusher_pairs['dy']**2
)

# Blocker facing direction unit vector under NFL angle convention:
# 0° = (0,1), 90° = (1,0)
o_rad = np.radians(blocker_rusher_pairs['o_blocker'])
blocker_rusher_pairs['ux_blocker'] = np.sin(o_rad)
blocker_rusher_pairs['uy_blocker'] = np.cos(o_rad)

# Unit vector from blocker to rusher
nonzero_dist = blocker_rusher_pairs['actual_distance'] > 0

blocker_rusher_pairs['ux_to_rusher'] = np.where(
    nonzero_dist,
    blocker_rusher_pairs['dx'] / blocker_rusher_pairs['actual_distance'],
    np.nan
)
blocker_rusher_pairs['uy_to_rusher'] = np.where(
    nonzero_dist,
    blocker_rusher_pairs['dy'] / blocker_rusher_pairs['actual_distance'],
    np.nan
)

# cos(theta) using dot product
blocker_rusher_pairs['cos_theta'] = (
    blocker_rusher_pairs['ux_blocker'] * blocker_rusher_pairs['ux_to_rusher']
    + blocker_rusher_pairs['uy_blocker'] * blocker_rusher_pairs['uy_to_rusher']
).clip(-1, 1)

# Optional: recover theta in degrees
blocker_rusher_pairs['theta_deg'] = np.degrees(
    np.arccos(blocker_rusher_pairs['cos_theta'])
)

# Weighted distance
blocker_rusher_pairs['weighted_distance'] = np.where(
    blocker_rusher_pairs['cos_theta'] > 0,
    blocker_rusher_pairs['actual_distance'] / blocker_rusher_pairs['cos_theta'],
    np.inf
)

# Optional final column selection
blocker_rusher_pairs = blocker_rusher_pairs[
    [
        'gameId', 'playId', 'frameID',
        'blocker_nflId', 'rusher_nflId',
        'x_blocker', 'y_blocker', 'o_blocker',
        'x_rusher', 'y_rusher',
        'dx', 'dy',
        'actual_distance', 'cos_theta', 'theta_deg', 'weighted_distance'
    ]
].copy()

blocker_rusher_pairs

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,dx,dy,actual_distance,cos_theta,theta_deg,weighted_distance
0,2021090900,97,11,40151,41263,41.58,24.31,71.03,42.34,19.20,0.76,-5.11,5.166208,-0.182416,100.510525,inf
1,2021090900,97,11,40151,42403,41.58,24.31,71.03,42.71,32.07,1.13,7.76,7.841843,0.457953,62.744905,17.123686
2,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,0.97,1.06,1.436837,0.878246,28.568545,1.636031
3,2021090900,97,11,40151,53441,41.58,24.31,71.03,42.81,22.24,1.23,-2.07,2.407862,0.203623,78.251096,11.825097
4,2021090900,97,11,40151,53504,41.58,24.31,71.03,43.25,26.78,1.67,2.47,2.981577,0.798984,36.966849,3.731712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3844771,2021110100,4433,36,52507,52585,22.41,26.22,217.53,21.04,20.71,-1.37,-5.51,5.677764,0.916592,23.567181,6.194429
3844772,2021110100,4433,37,52507,42406,22.14,26.15,207.07,21.81,24.27,-0.33,-1.88,1.908743,0.955720,17.114186,1.997178
3844773,2021110100,4433,37,52507,43326,22.14,26.15,207.07,22.17,17.56,0.03,-8.59,8.590052,0.888856,27.270101,9.664162
3844774,2021110100,4433,37,52507,43338,22.14,26.15,207.07,21.73,25.86,-0.41,-0.29,0.502195,0.885738,27.657579,0.566980


In [22]:
blocker_rusher_pairs = blocker_rusher_pairs[
    (blocker_rusher_pairs['weighted_distance'] >= 0) &
    (blocker_rusher_pairs['weighted_distance'] <= 3.5)
].copy()

closest_rusher_per_blocker = (
    blocker_rusher_pairs
    .sort_values(
        ['gameId', 'playId', 'frameID', 'blocker_nflId', 'weighted_distance']
    )
    .drop_duplicates(
        subset=['gameId', 'playId', 'frameID', 'blocker_nflId'],
        keep='first'
    )
    .copy()
)

In [23]:
closest_rusher_per_blocker

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,dx,dy,actual_distance,cos_theta,theta_deg,weighted_distance
2,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,0.97,1.06,1.436837,0.878246,28.568545,1.636031
132,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,2.09,-1.93,2.844820,0.973382,13.249226,2.922613
262,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,0.96,-0.30,1.005783,0.988824,8.574025,1.017151
393,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,1.66,-0.46,1.722556,0.998252,3.388501,1.725573
523,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,2.21,1.26,2.543954,0.958078,16.649010,2.655268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3844770,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,-0.50,-0.16,0.524976,0.821892,34.725328,0.638741
3844151,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,0.29,-0.73,0.785493,0.999920,0.724041,0.785556
3844356,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,-0.23,0.50,0.550364,0.974222,13.037570,0.564926
3844669,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,0.08,-2.49,2.491285,0.979960,11.489805,2.542231


In [24]:
players = pd.read_csv("../cleaned_csv/players_cleaned.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")

In [25]:
merged_df = closest_rusher_per_blocker.merge(
    filtered_df[['gameId', 'playId', 'frameID', 'nflId','quarter','gameClock']],
    left_on=['gameId', 'playId', 'frameID', 'blocker_nflId'],
    right_on=['gameId', 'playId', 'frameID', 'nflId'],
    how='left'
).drop(columns=['nflId'])


# Add blocker_name and rusher_name from players
name_map = players[['nflId', 'displayName']].drop_duplicates()

merged_df = merged_df.merge(
    name_map.rename(columns={
        'nflId': 'blocker_nflId',
        'displayName': 'blocker_name'
    }),
    on='blocker_nflId',
    how='left'
)

merged_df = merged_df.merge(
    name_map.rename(columns={
        'nflId': 'rusher_nflId',
        'displayName': 'rusher_name'
    }),
    on='rusher_nflId',
    how='left'
)

# Add blocker_position and rusher_position from pffScoutingData using pff_positionLinedUp
pff_positions = pffScoutingData[
    ['gameId', 'playId', 'nflId', 'pff_positionLinedUp']
].drop_duplicates()

merged_df = merged_df.merge(
    pff_positions.rename(columns={
        'nflId': 'blocker_nflId',
        'pff_positionLinedUp': 'blocker_position'
    }),
    on=['gameId', 'playId', 'blocker_nflId'],
    how='left'
)

merged_df = merged_df.merge(
    pff_positions.rename(columns={
        'nflId': 'rusher_nflId',
        'pff_positionLinedUp': 'rusher_position'
    }),
    on=['gameId', 'playId', 'rusher_nflId'],
    how='left'
)

In [26]:
merged_df['rusher_position'].value_counts()

rusher_position
DRT      125124
DLT      114355
LEO       87759
LE        87426
REO       83561
ROLB      70326
LOLB      65632
RE        64810
NT        32778
NRT       18578
NLT       15293
RILB       9217
LILB       9195
MLB        3649
LLB        3202
RLB        3140
SCBL       1009
SCBR        927
SCBiL       282
SCBoL       241
SCBiR       238
LCB         219
RCB         155
SCBoR       124
SSR          12
Name: count, dtype: int64

In [28]:
# pd.read_csv("../outputs_csv/blocker_rusher_matchups.csv")
# base_filtered.iloc[:,10:]
merged_df

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
0,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,...,1.436837,0.878246,28.568545,1.636031,1,13:33,Ryan Jensen,Carlos Watkins,C,DRT
1,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,...,2.844820,0.973382,13.249226,2.922613,1,13:33,Donovan Smith,Carlos Watkins,LT,DRT
2,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,...,1.005783,0.988824,8.574025,1.017151,1,13:33,Ali Marpet,Carlos Watkins,LG,DRT
3,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,...,1.722556,0.998252,3.388501,1.725573,1,13:33,Alex Cappa,Micah Parsons,RG,LILB
4,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,...,2.543954,0.958078,16.649010,2.655268,1,13:33,Tristan Wirfs,Micah Parsons,RT,LILB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797247,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,...,0.524976,0.821892,34.725328,0.638741,4,0:35,Matt Peart,Jarran Reed,LT,RE
797248,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,...,0.785493,0.999920,0.724041,0.785556,4,0:35,Nate Solder,Michael Danna,RT,LEO
797249,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,...,0.550364,0.974222,13.037570,0.564926,4,0:35,Matt Skura,Frank Clark,LG,ROLB
797250,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,...,2.491285,0.979960,11.489805,2.542231,4,0:35,Will Hernandez,Chris Jones,RG,LE


In [27]:
merged_df[(merged_df['gameId'] == 2021093000) & (merged_df['playId'] == 621)].iloc[0:50]

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
317707,2021093000,621,11,41322,43455,63.25,27.67,89.22,64.91,27.63,...,1.660482,0.999289,2.160354,1.661663,1,3:25,Brandon Linder,D.J. Reader,C,NT
317708,2021093000,621,11,41939,48145,62.99,29.64,73.93,64.68,30.84,...,2.072704,0.943761,19.307006,2.196218,1,3:25,Andrew Norwell,Wyatt Ray,LG,RE
317709,2021093000,621,11,42410,43455,62.97,26.12,64.55,64.91,27.63,...,2.458394,0.976502,12.445425,2.517552,1,3:25,A.J. Cann,D.J. Reader,RG,NT
317710,2021093000,621,11,44846,48145,62.63,31.25,79.92,64.68,30.84,...,2.090598,0.931120,21.389932,2.245251,1,3:25,Cam Robinson,Wyatt Ray,LT,RE
317711,2021093000,621,11,47818,46138,62.20,24.60,98.50,64.66,25.28,...,2.552254,0.913886,23.952011,2.792749,1,3:25,Jawaan Taylor,B.J. Hill,RT,LE
317712,2021093000,621,12,41322,43455,63.16,27.64,93.16,64.72,27.57,...,1.561570,0.999947,0.590759,1.561653,1,3:25,Brandon Linder,D.J. Reader,C,NT
317713,2021093000,621,12,41939,48145,62.97,29.73,73.93,64.52,30.86,...,1.918176,0.939554,20.023256,2.041582,1,3:25,Andrew Norwell,Wyatt Ray,LG,RE
317714,2021093000,621,12,42410,43455,62.95,26.00,69.55,64.72,27.57,...,2.365967,0.932808,21.123207,2.536393,1,3:25,A.J. Cann,D.J. Reader,RG,NT
317715,2021093000,621,12,44846,48145,62.51,31.32,77.88,64.52,30.86,...,2.061965,0.906230,25.010481,2.275321,1,3:25,Cam Robinson,Wyatt Ray,LT,RE
317716,2021093000,621,12,47818,46138,62.02,24.55,98.50,64.48,25.22,...,2.549608,0.915414,23.735409,2.785196,1,3:25,Jawaan Taylor,B.J. Hill,RT,LE


In [ ]:
merged_df.columns

In [ ]:
merged_df[merged_df['gameId'] == 2021093000].drop_duplicates(subset=['gameId','playId'])

In [29]:


if base_filtered.drop_duplicates(subset=['gameId','playId']).shape[0] != 7312:
    print('here')
    base_filtered = base_filtered.merge(
        merged_df[['gameId', 'playId']].drop_duplicates(),
        on=['gameId', 'playId'],
        how='inner'
    )

    base_filtered = base_filtered.merge(
        filtered_df[['gameId', 'playId', 'frameIdBallSnap']].drop_duplicates(['gameId', 'playId']),
        on=['gameId', 'playId'],
        how='left'
    )
    base_filtered = base_filtered.drop(['ttt_le_1_5_and_behind_los'],axis=1)
    base_filtered=base_filtered.rename(columns={'frameID':'frameId'})


here


In [30]:
blocker_rusher_output_df=merged_df.rename(columns={'frameID':'frameId'})
blocker_rusher_output_df.to_csv("../outputs_csv/blocker_rusher_matchups.csv", index=False)
base_filtered.to_csv("../outputs_csv/base_filtered.csv", index=False)


In [31]:
base_filtered.drop_duplicates(subset=['gameId','playId'])

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,x,y,s,a,dis,o,dir,event,frameIdEndWindow,frameIdBallSnap
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36,6
792,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:09.900,...,106.92,20.83,0.07,0.08,0.01,153.18,250.65,NaN,32,7
1496,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.000,...,75.23,28.60,0.27,0.54,0.03,131.50,46.10,line_set,28,6
2112,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:51.600,...,48.71,28.12,0.20,0.19,0.03,88.15,43.07,NaN,36,6
2904,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.000,...,53.50,36.53,0.03,0.03,0.03,85.90,72.33,NaN,33,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5258770,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.300,...,18.90,47.67,0.00,0.00,0.00,241.60,341.47,NaN,37,7
5259584,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:40.400,...,33.29,26.85,0.00,0.00,0.00,86.74,34.36,NaN,37,7
5260398,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.300,...,36.31,20.84,0.00,0.00,0.00,89.16,353.34,NaN,37,7
5261212,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:39.600,...,27.97,20.56,0.01,0.01,0.01,92.81,250.80,NaN,33,7


In [32]:
blocker_rusher_output_df

,gameId,playId,frameId,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
0,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,...,1.436837,0.878246,28.568545,1.636031,1,13:33,Ryan Jensen,Carlos Watkins,C,DRT
1,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,...,2.844820,0.973382,13.249226,2.922613,1,13:33,Donovan Smith,Carlos Watkins,LT,DRT
2,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,...,1.005783,0.988824,8.574025,1.017151,1,13:33,Ali Marpet,Carlos Watkins,LG,DRT
3,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,...,1.722556,0.998252,3.388501,1.725573,1,13:33,Alex Cappa,Micah Parsons,RG,LILB
4,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,...,2.543954,0.958078,16.649010,2.655268,1,13:33,Tristan Wirfs,Micah Parsons,RT,LILB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797247,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,...,0.524976,0.821892,34.725328,0.638741,4,0:35,Matt Peart,Jarran Reed,LT,RE
797248,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,...,0.785493,0.999920,0.724041,0.785556,4,0:35,Nate Solder,Michael Danna,RT,LEO
797249,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,...,0.550364,0.974222,13.037570,0.564926,4,0:35,Matt Skura,Frank Clark,LG,ROLB
797250,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,...,2.491285,0.979960,11.489805,2.542231,4,0:35,Will Hernandez,Chris Jones,RG,LE
